# Deep Agents: Building Complex Agents for Long-Horizon Tasks

In this notebook, we'll explore **Deep Agents** - a new approach to building AI agents that can handle complex, multi-step tasks over extended periods. We'll implement all four key elements of Deep Agents while building on our Personal Wellness Assistant use case.

**Learning Objectives:**
- Understand the four key elements of Deep Agents: Planning, Context Management, Subagent Spawning, and Long-term Memory
- Implement each element progressively using the `deepagents` package
- Learn to use Skills for progressive capability disclosure
- Use the `deepagents-cli` for interactive agent sessions

## Table of Contents:

- **Breakout Room #1:** Deep Agent Foundations
  - Task 1: Dependencies & Setup
  - Task 2: Understanding Deep Agents
  - Task 3: Planning with Todo Lists
  - Task 4: Context Management with File Systems
  - Task 5: Basic Deep Agent
  - Question #1 & Question #2
  - Activity #1: Build a Research Agent

- **Breakout Room #2:** Advanced Features & Integration
  - Task 6: Subagent Spawning
  - Task 7: Long-term Memory Integration
  - Task 8: Skills - On-Demand Capabilities
  - Task 9: Using deepagents-cli
  - Task 10: Building a Complete Deep Agent System
  - Question #3 & Question #4
  - Activity #2: Build a Wellness Coach Agent

---
# 🤝 Breakout Room #1
## Deep Agent Foundations

## Task 1: Dependencies & Setup

Before we begin, make sure you have:

1. **API Keys** for:
   - Anthropic (default for Deep Agents) or OpenAI
   - LangSmith (optional, for tracing)
   - Tavily (optional, for web search)

2. **Dependencies installed** via `uv sync`

3. **For the CLI** (Task 9): `uv pip install deepagents-cli`

### Environment Setup

You can either:
- Create a `.env` file with your API keys (recommended):
  ```
  ANTHROPIC_API_KEY=your_key_here
  OPENAI_API_KEY=your_key_here
  LANGCHAIN_API_KEY=your_key_here
  ```
- Or enter them interactively when prompted

In [2]:
# Core imports
import os
import getpass
from uuid import uuid4
from typing import Annotated, TypedDict, Literal

import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

def get_api_key(env_var: str, prompt: str) -> str:
    """Get API key from environment or prompt user."""
    value = os.environ.get(env_var, "")
    if not value:
        value = getpass.getpass(prompt)
        if value:
            os.environ[env_var] = value
    return value

In [3]:
# Set Anthropic API Key (default for Deep Agents)
anthropic_key = get_api_key("ANTHROPIC_API_KEY", "Anthropic API Key: ")
if anthropic_key:
    print("Anthropic API key set")
else:
    print("Warning: No Anthropic API key configured")

In [4]:
# Optional: OpenAI for alternative models and subagents
openai_key = get_api_key("OPENAI_API_KEY", "OpenAI API Key (press Enter to skip): ")
if openai_key:
    print("OpenAI API key set")
else:
    print("OpenAI API key not configured (optional)")

OpenAI API key set


In [5]:
# Optional: LangSmith for tracing
langsmith_key = get_api_key("LANGCHAIN_API_KEY", "LangSmith API Key (press Enter to skip): ")

if langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = f"AIE9 - Deep Agents - {uuid4().hex[0:8]}"
    print(f"LangSmith tracing enabled. Project: {os.environ['LANGCHAIN_PROJECT']}")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing disabled")

LangSmith tracing disabled


In [6]:
# Verify deepagents installation
from deepagents import create_deep_agent
print("deepagents package imported successfully!")

# Test with a simple agent
test_agent = create_deep_agent(model="openai:gpt-4o-mini")
result = test_agent.invoke({
    "messages": [{"role": "user", "content": "Say 'Deep Agents ready!' in exactly those words."}]
})
print(result["messages"][-1].content)

deepagents package imported successfully!
Deep Agents ready!


## Task 2: Understanding Deep Agents

**Deep Agents** represent a shift from simple tool-calling loops to sophisticated agents that can handle complex, long-horizon tasks. They address four key challenges:

### The Four Key Elements

| Element | Challenge Addressed | Implementation |
|---------|---------------------|----------------|
| **Planning** | "What should I do?" | Todo lists that persist task state |
| **Context Management** | "What do I know?" | File systems for storing/retrieving info |
| **Subagent Spawning** | "Who can help?" | Task tool for delegating to specialists |
| **Long-term Memory** | "What did I learn?" | LangGraph Store for cross-session memory |

### Deep Agents vs Traditional Agents

```
Traditional Agent Loop:
┌─────────────────────────────────────┐
│  User Query                         │
│       ↓                             │
│  Think → Act → Observe → Repeat     │
│       ↓                             │
│  Response                           │
└─────────────────────────────────────┘
Problems: Context bloat, no delegation,
          loses track of complex tasks

Deep Agent Architecture:
┌─────────────────────────────────────────────────────────┐
│                    Deep Agent                           │
├─────────────────────────────────────────────────────────┤
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐   │
│  │   PLANNING   │  │   CONTEXT    │  │   MEMORY     │   │
│  │              │  │  MANAGEMENT  │  │              │   │
│  │ write_todos  │  │              │  │   Store      │   │
│  │ update_todo  │  │  read_file   │  │  namespace   │   │
│  │ list_todos   │  │  write_file  │  │  get/put     │   │
│  │              │  │  edit_file   │  │              │   │
│  └──────────────┘  │  ls          │  └──────────────┘   │
│                    └──────────────┘                     │
│  ┌──────────────────────────────────────────────────┐   │
│  │              SUBAGENT SPAWNING                   │   │
│  │                                                  │   │
│  │  task(prompt, tools, model, system_prompt)       │   │
│  │       ↓              ↓              ↓            │   │
│  │  ┌────────┐    ┌────────┐    ┌────────┐          │   │
│  │  │Research│    │Writing │    │Analysis│          │   │
│  │  │Subagent│    │Subagent│    │Subagent│          │   │
│  │  └────────┘    └────────┘    └────────┘          │   │
│  └──────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────┘
```

### When to Use Deep Agents

| Use Case | Traditional Agent | Deep Agent |
|----------|-------------------|------------|
| Simple Q&A | ✅ | Overkill |
| Single-step tool use | ✅ | Overkill |
| Multi-step research | ⚠️ May lose track | ✅ |
| Complex projects | ❌ Context overflow | ✅ |
| Parallel task execution | ❌ | ✅ |
| Long-running sessions | ❌ | ✅ |

### Key Insight: "Planning is Context Engineering"

Deep Agents treat planning not as a separate phase, but as **context engineering**:
- Todo lists aren't just task trackers—they're **persistent context** about what to do
- File systems aren't just storage—they're **extended memory** beyond the context window
- Subagents aren't just helpers—they're **context isolation** to prevent bloat

## Task 3: Planning with Todo Lists

The first key element of Deep Agents is **Planning**. Instead of trying to hold all task state in the conversation, Deep Agents use structured todo lists.

### Why Todo Lists?

1. **Persistence**: Tasks survive across conversation turns
2. **Visibility**: Both agent and user can see progress
3. **Structure**: Clear tracking of what's done vs pending
4. **Recovery**: Agent can resume from where it left off

### Todo List Tools

| Tool | Purpose |
|------|----------|
| `write_todos` | Create a structured task list |
| `update_todo` | Mark tasks as complete/in-progress |
| `list_todos` | View current task state |

In [7]:
from langchain_core.tools import tool
from typing import List, Optional
import json

# Simple in-memory todo storage for demonstration
# In production, Deep Agents use persistent storage
TODO_STORE = {}

@tool
def write_todos(todos: List[dict]) -> str:
    """Create a list of todos for tracking task progress.
    
    Args:
        todos: List of todo items, each with 'title' and optional 'description'
    
    Returns:
        Confirmation message with todo IDs
    """
    created = []
    for i, todo in enumerate(todos):
        todo_id = f"todo_{len(TODO_STORE) + i + 1}"
        TODO_STORE[todo_id] = {
            "id": todo_id,
            "title": todo.get("title", "Untitled"),
            "description": todo.get("description", ""),
            "status": "pending"
        }
        created.append(todo_id)
    return f"Created {len(created)} todos: {', '.join(created)}"

@tool
def update_todo(todo_id: str, status: Literal["pending", "in_progress", "completed"]) -> str:
    """Update the status of a todo item.
    
    Args:
        todo_id: The ID of the todo to update
        status: New status (pending, in_progress, completed)
    
    Returns:
        Confirmation message
    """
    if todo_id not in TODO_STORE:
        return f"Todo {todo_id} not found"
    TODO_STORE[todo_id]["status"] = status
    return f"Updated {todo_id} to {status}"

@tool
def list_todos() -> str:
    """List all todos with their current status.
    
    Returns:
        Formatted list of all todos
    """
    if not TODO_STORE:
        return "No todos found"
    
    result = []
    for todo_id, todo in TODO_STORE.items():
        status_emoji = {"pending": "⬜", "in_progress": "🔄", "completed": "✅"}
        emoji = status_emoji.get(todo["status"], "❓")
        result.append(f"{emoji} [{todo_id}] {todo['title']} ({todo['status']})")
    return "\n".join(result)

print("Todo tools defined!")

Todo tools defined!


In [8]:
# Test the todo tools
TODO_STORE.clear()  # Reset for demo

# Create some wellness todos
result = write_todos.invoke({
    "todos": [
        {"title": "Assess current sleep patterns", "description": "Review user's sleep schedule and quality"},
        {"title": "Research sleep improvement strategies", "description": "Find evidence-based techniques"},
        {"title": "Create personalized sleep plan", "description": "Combine findings into actionable steps"},
    ]
})
print(result)
print("\nCurrent todos:")
print(list_todos.invoke({}))

Created 3 todos: todo_1, todo_3, todo_5

Current todos:
⬜ [todo_1] Assess current sleep patterns (pending)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


In [9]:
# Simulate progress
update_todo.invoke({"todo_id": "todo_1", "status": "completed"})
update_todo.invoke({"todo_id": "todo_2", "status": "in_progress"})

print("After updates:")
print(list_todos.invoke({}))

After updates:
✅ [todo_1] Assess current sleep patterns (completed)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


## Task 4: Context Management with File Systems

The second key element is **Context Management**. Deep Agents use file systems to:

1. **Offload large content** - Store research, documents, and results to disk
2. **Persist across sessions** - Files survive beyond conversation context
3. **Share between subagents** - Subagents can read/write shared files
4. **Prevent context overflow** - Large tool results automatically saved to disk

### Automatic Context Management

Deep Agents automatically handle context limits:
- **Large result offloading**: Tool results >20k tokens → saved to disk
- **Proactive offloading**: At 85% context capacity → agent saves state to disk
- **Summarization**: Long conversations get summarized while preserving intent

### File System Tools

| Tool | Purpose |
|------|----------|
| `ls` | List directory contents |
| `read_file` | Read file contents |
| `write_file` | Create/overwrite files |
| `edit_file` | Make targeted edits |

In [10]:
import os
from pathlib import Path

# Create a workspace directory for our agent
WORKSPACE = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

@tool
def ls(path: str = ".") -> str:
    """List contents of a directory.
    
    Args:
        path: Directory path to list (default: current directory)
    
    Returns:
        List of files and directories
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"Directory not found: {path}"
    
    items = []
    for item in sorted(target.iterdir()):
        prefix = "[DIR]" if item.is_dir() else "[FILE]"
        size = f" ({item.stat().st_size} bytes)" if item.is_file() else ""
        items.append(f"{prefix} {item.name}{size}")
    
    return "\n".join(items) if items else "(empty directory)"

@tool
def read_file(path: str) -> str:
    """Read contents of a file.
    
    Args:
        path: Path to the file to read
    
    Returns:
        File contents
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    return target.read_text()

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file (creates or overwrites).
    
    Args:
        path: Path to the file to write
        content: Content to write to the file
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"Wrote {len(content)} characters to {path}"

@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """Edit a file by replacing text.
    
    Args:
        path: Path to the file to edit
        old_text: Text to find and replace
        new_text: Replacement text
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    
    content = target.read_text()
    if old_text not in content:
        return f"Text not found in {path}"
    
    new_content = content.replace(old_text, new_text, 1)
    target.write_text(new_content)
    return f"Updated {path}"

print("File system tools defined!")
print(f"Workspace: {WORKSPACE.absolute()}")

File system tools defined!
Workspace: c:\Users\llukacevic2\Projects\ai_bootcamp\AIE9\07_Deep_Agents\workspace


In [11]:
# Test the file system tools
print("Current workspace contents:")
print(ls.invoke({"path": "."}))

Current workspace contents:
(empty directory)


In [12]:
# Create a research notes file
notes = """# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations
"""

result = write_file.invoke({"path": "research/sleep_notes.md", "content": notes})
print(result)

# Verify it was created
print("\nResearch directory:")
print(ls.invoke({"path": "research"}))

Wrote 242 characters to research/sleep_notes.md

Research directory:
[FILE] sleep_notes.md (252 bytes)


In [13]:
# Read and edit the file
print("File contents:")
print(read_file.invoke({"path": "research/sleep_notes.md"}))

File contents:
# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations



## Task 5: Basic Deep Agent

Now let's create a basic Deep Agent using the `deepagents` package. This combines:
- Planning (todo lists)
- Context management (file system)
- A capable LLM backbone

### Configuring the FilesystemBackend

Deep Agents come with **built-in file tools** (`ls`, `read_file`, `write_file`, `edit_file`). To control where files are stored, we configure a `FilesystemBackend`:

```python
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(
    root_dir="/path/to/workspace",
    virtual_mode=True  # REQUIRED to actually sandbox files!
)
```

**Critical: `virtual_mode=True`**
- Without `virtual_mode=True`, agents can still write anywhere on the filesystem!
- The `root_dir` alone does NOT restrict file access
- `virtual_mode=True` blocks paths with `..`, `~`, and absolute paths outside root

In [14]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Configure the filesystem backend to use our workspace directory
# IMPORTANT: virtual_mode=True is required to actually restrict paths to root_dir
# Without it, agents can still write anywhere on the filesystem!
workspace_path = Path("workspace").absolute()
filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

# Combine our custom tools (for todo tracking)
# Note: Deep Agents has built-in file tools (ls, read_file, write_file, edit_file)
# that will use the configured FilesystemBackend
custom_tools = [
    write_todos,
    update_todo,
    list_todos,
]

# Create a basic Deep Agent
wellness_agent = create_deep_agent(
    model=init_chat_model("openai:gpt-4o-mini"),
    tools=custom_tools,
    backend=filesystem_backend,  # Configure where files are stored
    system_prompt="""You are a Personal Wellness Assistant that helps users improve their health.

When given a complex task:
1. First, create a todo list to track your progress
2. Work through each task, updating status as you go
3. Save important findings to files for reference
4. Provide a clear summary when complete

Be thorough but concise. Always explain your reasoning."""
)

print(f"Basic Deep Agent created!")
print(f"File operations sandboxed to: {workspace_path}")

Basic Deep Agent created!
File operations sandboxed to: c:\Users\llukacevic2\Projects\ai_bootcamp\AIE9\07_Deep_Agents\workspace


In [15]:
# Reset todo store for fresh demo
TODO_STORE.clear()

# Test with a multi-step wellness task
result = wellness_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please create a personalized sleep improvement plan for me and save it to a file."""
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
I have created a personalized sleep improvement plan based on your current habits and challenges. The plan is saved as `sleep_improvement_plan.txt` and includes the following steps:

1. **Analyze Current Sleep Habits**: Review inconsistencies in bedtimes (10pm-1am), phone use, and feelings of tiredness in the morning.
  
2. **Establish a Consistent Bedtime**: Aim to go to bed consistently between 10pm and 11pm each night.
   
3. **Create a Pre-Sleep Routine**: Develop a calming pre-sleep routine that includes relaxing activities (e.g., reading, meditation) and minimizes screen time.
   
4. **Limit Phone Use in Bed**: Set a rule to stop using your phone at least 30 minutes before bedtime to reduce exposure to blue light.
   
5. **Implement a Wake-Up Routine**: Establish a morning routine that enhances alertness upon waking, such as stretching, drinking water, or exposure to natural light.
   
6. **Track Sleep Quality**: Use a sleep journal or an app to monitor sleep patt

---
## ❓ Question #1:

What are the **trade-offs** of using todo lists for planning? Consider:
- When might explicit planning overhead slow things down?
- How granular should todo items be?
- What happens if the agent creates todos but never completes them?

##### Answer:
Todo lists introduce structure and visibility into an agent’s reasoning process, but they also add coordination overhead and potential failure modes. Explicit planning can slow things down when tasks are simple or short lived or the evironment is dynamin and conditions change very frequently which makes it more expensive to maintain and update. Granularity should be well balanced (not too coarse and not too fine either) - each todo item should represent a meaningful, independently completable step with a clear success condition. If agent never finishes todos it can lead to execution drift, accumulated technical debt, loss of truth. 

## ❓ Question #2:

How would you design a **context management strategy** for a wellness agent that:
- Needs to reference a large health document (16KB)
- Tracks user metrics over time
- Must remember user conditions (allergies, medications) for safety

What goes in files vs. in the prompt? What should never be offloaded?

##### Answer:
A good context management strategy for a wellness agent should clearly separate long-term storage, safety-critical memory, and short-term prompt context. The full 16KB health document should not be placed directly into the prompt every time the agent runs. Instead, it should be stored as a file and indexed using a retrieval mechanism (such as chunking and embeddings), so that only the most relevant sections are fetched and inserted into the prompt when needed. User metrics tracked over time, such as sleep, mood, or activity levels, should be stored in a structured database rather than in the prompt. The agent can then generate summaries or trends (for example, “average sleep over the last week”) and include only those summaries in the prompt during each interaction. This prevents the prompt from growing uncontrollably while still preserving historical insight. Safety-critical information such as allergies, medications, and medical conditions must be stored persistently in a secure and structured form. The agent should never rely only on conversational context to remember these facts. Instead, a short safety summary derived from the structured data can be inserted into the prompt for every relevant interaction. This ensures consistent safety checks and reduces the risk of forgetting important constraints. Sensitive or regulated information should never be offloaded casually into prompts, logs, or public vector stores. Raw medical records, personal identifiers, authentication credentials, and other high-risk data must remain securely stored with proper access controls.

---
## 🏗️ Activity #1: Build a Research Agent

Build a Deep Agent that can research a wellness topic and produce a structured report.

### Requirements:
1. Create todos for the research process
2. Read from the HealthWellnessGuide.txt in the data folder
3. Save findings to a structured markdown file
4. Update todo status as tasks complete

### Test prompt:
"Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."

In [16]:
### YOUR CODE HERE ###

# Step 1: Create a research agent with appropriate tools
# Hint: You'll need file tools to read the wellness guide


# Step 2: Add a tool to read from the data folder
# Hint: Use Path("data/HealthWellnessGuide.txt")
@tool
def read_wellness_guide() -> str:
    """Read the health and wellness guide for research."""
    guide_path = Path("data/HealthWellnessGuide.txt")
    if not guide_path.exists():
        return "Wellness guide not found at data/HealthWellnessGuide.txt"
    return guide_path.read_text()

tools = [
    write_todos,
    update_todo,
    list_todos,
    read_wellness_guide
]


# Step 3: Create the agent with a research-focused system prompt
research_agent = create_deep_agent(
    model=init_chat_model("openai:gpt-5"),
    tools=tools,
    backend=filesystem_backend,  
    system_prompt=""" You are a Research Assistant specializing in health and wellness.
When given a research task:
1. Create a todo list to track your research progress
2. Work through each task, updating status as you go
3. Use the provided tools to gather information and organize your findings
4. Save important findings to a structured markdown file for reference (inside research/)
5. Provide a clear summary of your research when complete
Always explain your reasoning and cite sources when possible.
    
    """
)

print(f"Research Agent created!")
print(f"File operations sandboxed to: {workspace_path}")

# Step 4: Test with the stress management research task
TODO_STORE.clear()

# Test with a multi-step wellness task
result = research_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."""
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Research Agent created!
File operations sandboxed to: c:\Users\llukacevic2\Projects\ai_bootcamp\AIE9\07_Deep_Agents\workspace
Agent response:
Here’s what I did and why
- I reviewed vetted material in our wellness guide and current clinical evidence (systematic reviews and guidelines) to select techniques with the strongest support, broad applicability, and good safety profiles.
- I created a comprehensive, step-by-step guide and saved it to: /research/stress_management_guide.md
- The guide includes dosing, adaptations, safety notes, a 4-week starter plan, and a way to track progress (daily 0–10 stress ratings and weekly PSS-10).

Summary of at least 5 evidence-based stress management strategies
1) Mindfulness-based training (e.g., MBSR)
- Why it works: Multiple meta-analyses show reductions in perceived stress, anxiety, and improved well-being across populations [Goyal 2014; Khoury 2013].
- How to do it: 10–20 minutes/day, 5–6 days/week. Mix breath-focused practice and body scan; consi

In [17]:
# Check what the agent created
print("Todo list after task:")
print(list_todos.invoke({}))

research = Path("workspace/research")
print("\n" + "="*50)
print("\nWorkspace contents:")
# List files in the workspace directory
for f in sorted(research.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    else:
        print(f"  [DIR] {f.name}/")

Todo list after task:
✅ [todo_1] Scan existing wellness guide content (completed)
✅ [todo_3] Outline evidence-based stress management strategies (completed)
✅ [todo_5] Draft comprehensive practical guide (completed)
✅ [todo_7] Cite sources (completed)
✅ [todo_9] Save guide to research/stress_management_guide.md (completed)
🔄 [todo_11] Provide summary of key findings to user (in_progress)


Workspace contents:
  [FILE] sleep_notes.md (252 bytes)
  [FILE] stress_management_guide.md (10603 bytes)


---
## ❓ Question #1:

What are the **trade-offs** of using todo lists for planning? Consider:
- When might explicit planning overhead slow things down?
- How granular should todo items be?
- What happens if the agent creates todos but never completes them?

##### Answer:
*Your answer here*

## ❓ Question #2:

How would you design a **context management strategy** for a wellness agent that:
- Needs to reference a large health document (16KB)
- Tracks user metrics over time
- Must remember user conditions (allergies, medications) for safety

What goes in files vs. in the prompt? What should never be offloaded?

##### Answer:
*Your answer here*

---
## 🏗️ Activity #1: Build a Research Agent

Build a Deep Agent that can research a wellness topic and produce a structured report.

### Requirements:
1. Create todos for the research process
2. Read from the HealthWellnessGuide.txt in the data folder
3. Save findings to a structured markdown file
4. Update todo status as tasks complete

### Test prompt:
"Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."

In [ ]:
### YOUR CODE HERE ###

# Step 1: Create a research agent with appropriate tools
# Hint: You'll need file tools to read the wellness guide


# Step 2: Add a tool to read from the data folder
# Hint: Use Path("data/HealthWellnessGuide.txt")


# Step 3: Create the agent with a research-focused system prompt


# Step 4: Test with the stress management research task


---
# 🤝 Breakout Room #2
## Advanced Features & Integration

## Task 6: Subagent Spawning

The third key element is **Subagent Spawning**. This allows a Deep Agent to delegate tasks to specialized subagents.

### Why Subagents?

1. **Context Isolation**: Each subagent has its own context window, preventing bloat
2. **Specialization**: Different subagents can have different tools/prompts
3. **Parallelism**: Multiple subagents can work simultaneously
4. **Cost Optimization**: Use cheaper models for simpler subtasks

### How Subagents Work

```
Main Agent
    ├── task("Research sleep science", model="gpt-4o-mini")
    │       └── Returns: Summary of findings
    │
    ├── task("Analyze user's sleep data", tools=[analyze_tool])
    │       └── Returns: Analysis results
    │
    └── task("Write recommendations", system_prompt="Be concise")
            └── Returns: Final recommendations
```

Key benefit: The main agent only receives **summaries**, not all the intermediate context!

In [18]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Define specialized subagent configurations
# Note: Subagents inherit the backend from the parent agent
research_subagent = {
    "name": "research-agent",
    "description": "Use this agent to research wellness topics in depth. It can read documents and synthesize information.",
    "system_prompt": """You are a wellness research specialist. Your job is to:
1. Find relevant information in provided documents
2. Synthesize findings into clear summaries
3. Cite sources when possible

Be thorough but concise. Focus on evidence-based information.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",  # Cheaper model for research
}

writing_subagent = {
    "name": "writing-agent",
    "description": "Use this agent to create well-structured documents, plans, and guides.",
    "system_prompt": """You are a wellness content writer. Your job is to:
1. Take research findings and turn them into clear, actionable content
2. Structure information for easy understanding
3. Use formatting (headers, bullets, etc.) effectively

Write in a supportive, encouraging tone.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

print("Subagent configurations defined!")

Subagent configurations defined!


In [19]:
# Create a coordinator agent that can spawn subagents
coordinator_agent = create_deep_agent(
    model=init_chat_model("openai:gpt-4o"),
    tools=[write_todos, update_todo, list_todos],
    backend=filesystem_backend,  # Use the same backend - subagents inherit it
    subagents=[research_subagent, writing_subagent],
    system_prompt="""You are a Wellness Project Coordinator. Your role is to:
1. Break down complex wellness requests into subtasks
2. Delegate research to the research-agent
3. Delegate content creation to the writing-agent
4. Coordinate the overall workflow using todos

Use subagents for specialized work rather than doing everything yourself.
This keeps the work organized and the results high-quality."""
)

print("Coordinator agent created with subagent capabilities!")

Coordinator agent created with subagent capabilities!


In [20]:
# Reset for demo
TODO_STORE.clear()

# Test the coordinator with a complex task
result = coordinator_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Create a comprehensive morning routine guide for better energy.
        
The guide should:
1. Research the science behind morning routines
2. Include practical steps for exercise, nutrition, and mindset
3. Be saved as a well-formatted markdown file"""
    }]
})

print("Coordinator response:")
print(result["messages"][-1].content)

Coordinator response:
I have created a comprehensive guide on morning routines aimed at enhancing energy levels. The guide integrates scientific research with practical steps across exercise, nutrition, and mindset. It has been saved as a markdown file titled [**morning_routine_guide.md**](sandbox:/morning_routine_guide.md).

Feel free to review the guide and let me know if there are any additional adjustments or topics you would like to explore!


In [21]:
# Check the results
print("Final todo status:")
print(list_todos.invoke({}))

print("\nGenerated files in workspace:")
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

Final todo status:
✅ [todo_1] Research science behind morning routines (completed)
✅ [todo_3] Develop practical steps for exercise, nutrition, and mindset (completed)
✅ [todo_5] Create a formatted markdown file (completed)

Generated files in workspace:
  [FILE] morning_routine_guide.md (3479 bytes)
  [DIR] research/
  [FILE] sleep_improvement_plan.txt (1058 bytes)


## Task 7: Long-term Memory Integration

The fourth key element is **Long-term Memory**. Deep Agents integrate with LangGraph's Store for persistent memory across sessions.

### Memory Types in Deep Agents

| Type | Scope | Use Case |
|------|-------|----------|
| **Thread Memory** | Single conversation | Current session context |
| **User Memory** | Across threads, per user | User preferences, history |
| **Shared Memory** | Across all users | Common knowledge, learned patterns |

### Integration with LangGraph Store

Deep Agents can use the same `InMemoryStore` (or `PostgresStore`) we learned in Session 6:

In [22]:
from langgraph.store.memory import InMemoryStore

# Create a memory store
memory_store = InMemoryStore()

# Store user profile
user_id = "user_alex"
profile_namespace = (user_id, "profile")

memory_store.put(profile_namespace, "name", {"value": "Alex"})
memory_store.put(profile_namespace, "goals", {
    "primary": "improve energy levels",
    "secondary": "better sleep"
})
memory_store.put(profile_namespace, "conditions", {
    "dietary": ["vegetarian"],
    "medical": ["mild anxiety"]
})
memory_store.put(profile_namespace, "preferences", {
    "exercise_time": "morning",
    "communication_style": "detailed"
})

print(f"Stored profile for {user_id}")

# Retrieve and display
for item in memory_store.search(profile_namespace):
    print(f"  {item.key}: {item.value}")

Stored profile for user_alex
  name: {'value': 'Alex'}
  goals: {'primary': 'improve energy levels', 'secondary': 'better sleep'}
  conditions: {'dietary': ['vegetarian'], 'medical': ['mild anxiety']}
  preferences: {'exercise_time': 'morning', 'communication_style': 'detailed'}


In [23]:
# Create memory-aware tools
from langgraph.store.base import BaseStore

@tool
def get_user_profile(user_id: str) -> str:
    """Retrieve a user's wellness profile from long-term memory.
    
    Args:
        user_id: The user's unique identifier
    
    Returns:
        User profile as formatted text
    """
    namespace = (user_id, "profile")
    items = list(memory_store.search(namespace))
    
    if not items:
        return f"No profile found for {user_id}"
    
    result = [f"Profile for {user_id}:"]
    for item in items:
        result.append(f"  {item.key}: {item.value}")
    return "\n".join(result)

@tool
def save_user_preference(user_id: str, key: str, value: str) -> str:
    """Save a user preference to long-term memory.
    
    Args:
        user_id: The user's unique identifier
        key: The preference key
        value: The preference value
    
    Returns:
        Confirmation message
    """
    namespace = (user_id, "preferences")
    memory_store.put(namespace, key, {"value": value})
    return f"Saved preference '{key}' for {user_id}"

print("Memory tools defined!")

Memory tools defined!


In [24]:
# Create a memory-enhanced agent
memory_tools = [
    get_user_profile,
    save_user_preference,
    write_todos,
    update_todo,
    list_todos,
]

memory_agent = create_deep_agent(
    model=init_chat_model("openai:gpt-4o"),
    tools=memory_tools,
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a Personal Wellness Assistant with long-term memory.

At the start of each conversation:
1. Check the user's profile to understand their goals and conditions
2. Personalize all advice based on their profile
3. Save any new preferences they mention

Always reference stored information to show you remember the user."""
)

print("Memory-enhanced agent created!")

Memory-enhanced agent created!


In [25]:
# Test the memory agent
TODO_STORE.clear()

result = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Hi! My user_id is user_alex. What exercise routine would you recommend for me?"
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Hi Alex! Based on your profile, here’s a personalized exercise routine to help you improve your energy levels and enhance your sleep quality. Since you prefer exercising in the morning, this routine is designed to kick-start your day:

**Morning Routine:**

1. **Warm-up (5-10 minutes):**
   - Dynamic stretches (arm circles, leg swings)
   - Light jogging or brisk walking

2. **Cardio (20 minutes):**
   - Cycling or jogging (adjust the intensity to your fitness level)
   - Focus on maintaining a steady pace to boost energy without over-exertion

3. **Strength Training (15 minutes):**
   - Bodyweight exercises like push-ups, squats, and lunges
   - Use light weights or resistance bands for added challenge if desired

4. **Yoga or Pilates (10-15 minutes):**
   - Focus on poses that promote relaxation and flexibility, such as downward dog and cat-cow stretches
   - Incorporate deep breathing exercises to help manage mild anxiety

5. **Cool Down (5 minutes):**
   - Static st

## Task 8: Skills - On-Demand Capabilities

**Skills** are a powerful feature for progressive capability disclosure. Instead of loading all tools upfront, agents can load specialized capabilities on demand.

### Why Skills?

1. **Context Efficiency**: Don't waste context on unused tool descriptions
2. **Specialization**: Skills can include detailed instructions for specific tasks
3. **Modularity**: Easy to add/remove capabilities
4. **Discoverability**: Agent can browse available skills

### SKILL.md Format

Skills are defined in markdown files with YAML frontmatter:

```markdown
---
name: skill-name
description: What this skill does
version: 1.0.0
tools:
  - tool1
  - tool2
---

# Skill Instructions

Detailed steps for how to use this skill...
```

In [26]:
# Let's look at the skills we created
skills_dir = Path("skills")

print("Available skills:")
for skill_dir in skills_dir.iterdir():
    if skill_dir.is_dir():
        skill_file = skill_dir / "SKILL.md"
        if skill_file.exists():
            content = skill_file.read_text()
            # Extract name and description from frontmatter
            lines = content.split("\n")
            name = ""
            desc = ""
            for line in lines:
                if line.startswith("name:"):
                    name = line.split(":", 1)[1].strip()
                if line.startswith("description:"):
                    desc = line.split(":", 1)[1].strip()
            print(f"  - {name}: {desc}")

Available skills:
  - meal-planning: Create personalized meal plans based on dietary needs and preferences
  - wellness-assessment: Assess user wellness goals and create personalized recommendations


In [27]:
# Read the wellness-assessment skill
skill_content = Path("skills/wellness-assessment/SKILL.md").read_text()
print(skill_content)

---
name: wellness-assessment
description: Assess user wellness goals and create personalized recommendations
version: 1.0.0
tools:
  - read_file
  - write_file
---

# Wellness Assessment Skill

You are conducting a comprehensive wellness assessment. Follow these steps:

## Step 1: Gather Information
Ask the user about:
- Current health goals (weight, fitness, stress, sleep)
- Any medical conditions or limitations
- Current exercise routine (or lack thereof)
- Dietary preferences and restrictions
- Sleep patterns and quality
- Stress levels and sources

## Step 2: Analyze Responses
Review the user's answers and identify:
- Primary wellness priority
- Secondary goals
- Potential barriers to success
- Existing healthy habits to build on

## Step 3: Create Assessment Report
Write a wellness assessment report to `workspace/wellness_assessment.md` containing:
- Summary of current wellness state
- Identified strengths
- Areas for improvement
- Recommended focus areas (prioritized)
- Suggeste

In [28]:
# Create a skill-aware tool
@tool
def load_skill(skill_name: str) -> str:
    """Load a skill's instructions for a specialized task.
    
    Available skills:
    - wellness-assessment: Assess user wellness and create recommendations
    - meal-planning: Create personalized meal plans
    
    Args:
        skill_name: Name of the skill to load
    
    Returns:
        Skill instructions
    """
    skill_path = Path(f"skills/{skill_name}/SKILL.md")
    if not skill_path.exists():
        available = [d.name for d in Path("skills").iterdir() if d.is_dir()]
        return f"Skill '{skill_name}' not found. Available: {', '.join(available)}"
    
    return skill_path.read_text()

print("Skill loader defined!")

Skill loader defined!


In [29]:
# Create an agent that can load and use skills
skill_agent = create_deep_agent(
    model=init_chat_model("openai:gpt-4o"),
    tools=[
        load_skill,
        write_todos,
        update_todo,
        list_todos,
    ],
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a wellness assistant with access to specialized skills.

When a user asks for something that matches a skill:
1. Load the appropriate skill using load_skill()
2. Follow the skill's instructions carefully
3. Save outputs as specified in the skill

Available skills:
- wellness-assessment: For comprehensive wellness evaluations
- meal-planning: For creating personalized meal plans

If no skill matches, use your general wellness knowledge."""
)

print("Skill-aware agent created!")

Skill-aware agent created!


In [30]:
# Test with a skill-appropriate request
TODO_STORE.clear()

result = skill_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "I'd like a wellness assessment. I'm a 35-year-old office worker who sits most of the day, has trouble sleeping, and wants to lose 15 pounds. I'm vegetarian and have no major health conditions."
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Your wellness assessment is complete! I've created a detailed report summarizing your current wellness state, strengths, areas for improvement, recommended focus areas, and suggested next steps.

Here is an overview of your personalized recommendations:

### Immediate Action Items (Start Today)
1. **Physical Activity**: Take short walking breaks every hour during work.
2. **Sleep Quality**: Establish a consistent bedtime and wake-up time.
3. **Nutrition**: Plan meals with high-protein plant-based ingredients like beans and lentils.

### Short-Term Goals (1-2 Weeks)
1. Start a simple low-impact exercise routine, such as walking or yoga, for at least 30 minutes a day.
2. Identify and create a wind-down routine to improve sleep quality before bed.
3. Replace processed foods with whole food alternatives at least once a day.

### Long-Term Goals (1-3 Months)
1. Progressively increase exercise intensity and frequency as tolerated.
2. Monitor sleep patterns and make environmen

## Task 9: Using deepagents-cli

The `deepagents-cli` provides an interactive terminal interface for working with Deep Agents.

### Installation

```bash
uv pip install deepagents-cli
# or
pip install deepagents-cli
```

### Key Features

| Feature | Description |
|---------|-------------|
| **Interactive Sessions** | Chat with your agent in the terminal |
| **Conversation Resume** | Pick up where you left off |
| **Human-in-the-Loop** | Approve or reject agent actions |
| **File System Access** | Agent can read/write to your filesystem |
| **Remote Sandboxing** | Run in isolated Docker containers |

### Basic Usage

```bash
# Start an interactive session
deepagents

# Resume a previous conversation
deepagents --resume

# Use a specific model
deepagents --model openai:gpt-4o

# Enable human-in-the-loop approval
deepagents --approval-mode full
```

### Example Session

```
$ deepagents

Welcome to Deep Agents CLI!

You: Create a 7-day meal plan for a vegetarian athlete

Agent: I'll create a comprehensive meal plan for you. Let me:
1. Research vegetarian athlete nutrition needs
2. Design balanced daily menus
3. Save the plan to a file

[Agent uses tools...]

Agent: I've created your meal plan! You can find it at:
workspace/vegetarian_athlete_meal_plan.md

You: /exit
```

In [31]:
# Check if CLI is installed
import subprocess

try:
    result = subprocess.run(["deepagents", "--version"], capture_output=True, text=True)
    print(f"deepagents-cli version: {result.stdout.strip()}")
except FileNotFoundError:
    print("deepagents-cli not installed. Install with:")
    print("  uv pip install deepagents-cli")
    print("  # or")
    print("  pip install deepagents-cli")

deepagents-cli version: deepagents 0.0.19


### Try It Yourself!

After installing the CLI, try these commands in your terminal:

```bash
# Basic interactive session
deepagents

# With a specific working directory
deepagents --workdir ./workspace

# See all options
deepagents --help
```

Sample prompts to try:
1. "Create a weekly workout plan and save it to a file"
2. "Research the health benefits of meditation and summarize in a report"
3. "Analyze my current diet and suggest improvements" (then provide details)

## Task 10: Building a Complete Deep Agent System

Now let's bring together all four elements to build a comprehensive "Wellness Coach" system:

1. **Planning**: Track multi-week wellness programs
2. **Context Management**: Store session notes and progress
3. **Subagent Spawning**: Delegate to specialists (exercise, nutrition, mindfulness)
4. **Long-term Memory**: Remember user preferences and history

In [32]:
# Define specialized wellness subagents
# Subagents inherit the backend from the parent, so they use the same workspace
exercise_specialist = {
    "name": "exercise-specialist",
    "description": "Expert in exercise science, workout programming, and physical fitness. Use for exercise-related questions and plan creation.",
    "system_prompt": """You are an exercise specialist with expertise in:
- Workout programming for different fitness levels
- Exercise form and safety
- Progressive overload principles
- Recovery and injury prevention

Always consider the user's fitness level and any physical limitations.
Provide clear, actionable exercise instructions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

nutrition_specialist = {
    "name": "nutrition-specialist",
    "description": "Expert in nutrition science, meal planning, and dietary optimization. Use for food-related questions and meal plans.",
    "system_prompt": """You are a nutrition specialist with expertise in:
- Macro and micronutrient balance
- Meal planning and preparation
- Dietary restrictions and alternatives
- Nutrition timing for performance

Always respect dietary restrictions and preferences.
Focus on practical, achievable meal suggestions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

mindfulness_specialist = {
    "name": "mindfulness-specialist",
    "description": "Expert in stress management, sleep optimization, and mental wellness. Use for stress, sleep, and mental health questions.",
    "system_prompt": """You are a mindfulness and mental wellness specialist with expertise in:
- Stress reduction techniques
- Sleep hygiene and optimization
- Meditation and breathing exercises
- Work-life balance strategies

Be supportive and non-judgmental.
Provide practical techniques that can be implemented immediately.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

print("Specialist subagents defined!")

Specialist subagents defined!


In [35]:
# Create the Wellness Coach coordinator
wellness_coach = create_deep_agent(
    model=init_chat_model("openai:gpt-4o"),
    tools=[
        # Planning
        write_todos,
        update_todo,
        list_todos,
        # Long-term Memory
        get_user_profile,
        save_user_preference,
        # Skills
        load_skill,
    ],
    backend=filesystem_backend,  # All file ops go to workspace
    subagents=[exercise_specialist, nutrition_specialist, mindfulness_specialist],
    system_prompt="""You are a Personal Wellness Coach that coordinates comprehensive wellness programs.

## Your Role
- Understand each user's unique goals, constraints, and preferences
- Create personalized, multi-week wellness programs
- Coordinate between exercise, nutrition, and mindfulness specialists
- Track progress and adapt recommendations

## Workflow
1. **Initial Assessment**: Get user profile and understand their situation
2. **Planning**: Create a todo list for the program components
3. **Delegation**: Use specialists for domain-specific content:
   - exercise-specialist: Workout plans and fitness guidance
   - nutrition-specialist: Meal plans and dietary advice
   - mindfulness-specialist: Stress and sleep optimization
4. **Integration**: Combine specialist outputs into a cohesive program
5. **Documentation**: Save all plans and recommendations to files

## Important
- Always check user profile first for context
- Respect any medical conditions or dietary restrictions
- Provide clear, actionable recommendations
- Save progress to files so users can reference later"""
)

print("Wellness Coach created with all 4 Deep Agent elements!")

Wellness Coach created with all 4 Deep Agent elements!


In [37]:
# Test the complete system
TODO_STORE.clear()

result = wellness_coach.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. I'd like you to create a 2-week wellness program for me.

I want to focus on:
1. Building a consistent exercise routine (I can exercise 3x per week for 30 mins)
2. Improving my diet (remember I'm vegetarian)
3. Better managing my work stress and improving my sleep

Please create comprehensive plans for each area and save them as separate files I can reference."""
    }]
})

print("Wellness Coach response:")
print(result["messages"][-1].content)

Wellness Coach response:
I've created a comprehensive 2-week wellness program for you, focusing on exercise, nutrition, and stress management. Here are the details and files for each plan:

1. **Exercise Plan**: Building a consistent routine with three 30-minute sessions per week.
   - [alex_exercise_plan.txt](sandbox:/wellness/alex_exercise_plan.txt)

2. **Meal Plan**: A balanced vegetarian plan designed to improve energy levels and sleep.
   - [alex_meal_plan.txt](sandbox:/wellness/alex_meal_plan.txt)

3. **Stress and Sleep Plan**: Techniques for managing mild anxiety and improving sleep quality.
   - [alex_stress_sleep_plan.txt](sandbox:/wellness/alex_stress_sleep_plan.txt)

Feel free to review these plans and let me know if you'd like any adjustments or have further preferences. Enjoy your wellness journey!


In [38]:
# Review what was created
print("=" * 60)
print("FINAL TODO STATUS")
print("=" * 60)
print(list_todos.invoke({}))

print("\n" + "=" * 60)
print("GENERATED FILES")
print("=" * 60)
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

FINAL TODO STATUS
No todos found

GENERATED FILES
  [DIR] 30_day_challenge/
  [FILE] meal_plan_veggie_energy_sleep.txt (3211 bytes)
  [FILE] morning_routine_guide.md (3479 bytes)
  [DIR] research/
  [FILE] sleep_improvement_plan.txt (1058 bytes)
  [DIR] wellness/
  [FILE] wellness_assessment.md (1738 bytes)
  [DIR] wellness_programs/


In [39]:
# Read one of the generated files
dir = Path("workspace/wellness_task10_alex")
files = list(dir.glob("*.txt"))
if files:
    print(f"\nContents of {files[0].name}:")
    print("=" * 60)
    print(files[0].read_text()[:2000] + "..." if len(files[0].read_text()) > 2000 else files[0].read_text())


Contents of alex_exercise_plan.txt:
Here's a 2-week exercise plan for Alex, focusing on building a consistent routine with three sessions per week. Each session will last 30 minutes and will include a mix of cardiovascular exercises, strength training, and flexibility work. The plan also considers vegetarian dietary constraints to support energy levels.

### Week 1

#### Day 1: Cardio & Core
- **Warm-Up** (5 minutes): 
  - March in place or light jogging
  - Arm circles and leg swings

- **Cardio** (20 minutes):
  - 5 minutes of brisk walking or light jogging
  - 5 minutes of jumping jacks
  - 5 minutes of high knees
  - 5 minutes of cooldown walking

- **Core Strengthening** (5 minutes):
  - Plank hold (20-30 seconds)
  - Bicycle crunches (15 reps)
  - Russian twists (10 reps each side)

---

#### Day 3: Strength Training
- **Warm-Up** (5 minutes):
  - Dynamic stretches (e.g., arm swings, leg swings)

- **Strength Exercises** (20 minutes):
  - Bodyweight squats (3 sets of 10 reps)
  

---
## ❓ Question #3:

What are the key considerations when designing **subagent configurations**?

Consider:
- When should subagents share tools vs have distinct tools?
- How do you decide which model to use for each subagent?
- What's the right granularity for subagent specialization?

##### Answer:
*Your answer here*

## ❓ Question #4:

For a **production wellness application** using Deep Agents, what would you need to add?

Consider:
- Safety guardrails for health advice
- Persistent storage (not in-memory)
- Multi-user support and isolation
- Monitoring and observability
- Cost management with subagents

##### Answer:
*Your answer here*

---
## 🏗️ Activity #2: Build a Wellness Coach Agent

Build your own wellness coach that uses all 4 Deep Agent elements.

### Requirements:
1. **Planning**: Create todos for a 30-day wellness challenge
2. **Context Management**: Store daily check-in notes
3. **Subagents**: At least 2 specialized subagents
4. **Memory**: Remember user preferences across interactions

### Challenge:
Create a "30-Day Wellness Challenge" system that:
- Generates a personalized 30-day plan
- Tracks daily progress
- Adapts recommendations based on feedback
- Saves a weekly summary report

In [45]:
### YOUR CODE HERE ###

# Step 1: Define your subagent configurations
physical_subagent = {
    "name": "physical-specialist",
    "description": "Expert in physical health, exercise science, and fitness programming. Use for exercise-related questions and plan creation.",
    "system_prompt": """You are a physical health specialist with expertise in:
- Exercise programming for different fitness levels
- Exercise form and safety
- Progressive overload principles
- Recovery and injury prevention
Always consider the user's fitness level and any physical limitations.
Provide clear, actionable exercise instructions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",  # Cheaper model
}

mental_subagent = {
    "name": "mental-specialist",
    "description": "Expert in mental wellness, stress management, and sleep optimization. Use for stress, sleep, and mental health questions.",
    "system_prompt": """You are a mental wellness specialist with expertise in:
- Stress reduction techniques
- Sleep hygiene and optimization
- Meditation and breathing exercises
- Work-life balance strategies
Be supportive and non-judgmental.
Provide practical techniques that can be implemented immediately.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",  # Cheaper model
}

# Step 2: Create any additional tools you need
@tool
def save_daily_ckeckin_notes(user_id: str, date: str, notes: str) -> str:
    """Store daily check-in notes for a user.
    
    Args:
        user_id: The user's unique identifier
        date: The date of the check-in (e.g., '2024-06-01')
        notes: The content of the check-in notes
    Returns:
        Confirmation message
    """
    user_namespace = (user_id, "daily_checkins")
    memory_store.put(user_namespace, date, {"value": notes})
    return f"Stored daily check-in for {user_id} on {date}"

@tool
def get_user_data(user_id: str, data_type: Literal["preferences", "daily_checkins"]) -> str:
    """Retrieve user data from long-term memory.
    
    Args:
        user_id: The user's unique identifier
        data_type: The type of data to retrieve (preferences or daily_checkins)
    
    Returns:
        Formatted user data
    """
    namespace = (user_id, data_type)
    items = list(memory_store.search(namespace))
    
    if not items:
        return f"No {data_type} found for {user_id}"
    
    result = [f"{data_type.capitalize()} for {user_id}:"]
    for item in items:
        result.append(f"  {item.key}: {item.value}")
    return "\n".join(result if items else f"No {data_type} found for {user_id}")

# Step 3: Build the main coordinator agent
wellness_coach_agent = create_deep_agent(
    model=init_chat_model("openai:gpt-5"),
    tools=[
        # Planning tools
        write_todos,
        update_todo,
        list_todos,
        # Memory tools
        get_user_profile,
        save_user_preference,
        get_user_data,
        save_daily_ckeckin_notes,
        # Skill tool
        load_skill,
    ],
    backend=filesystem_backend,  # Use the same backend - subagents inherit it
    subagents=[physical_subagent, mental_subagent],
    system_prompt="""You are a Personal Wellness Coach that coordinates comprehensive wellness programs.
## Your Role
- Understand each user's unique goals, constraints, and preferences
- Create personalized, 30 day wellness challenge
- Coordinate between physical and mental wellness specialists
- Track progress and adapt recommendations

## Workflow
1. **Initial Assessment**: Get user profile and preferences (if any) to understand their situation
2. **Planning**: Create a todo list for the program components (30 day wellness challenge)
3. **Delegation**: Use specialists for domain-specific content:
    - physical-specialist: Workout plans and fitness guidance
    - mental-specialist: Stress and sleep optimization
4. **Integration**: Combine specialist outputs into a cohesive program
5. **Documentation**: Save all plans and recommendations to .md files for reference (into '30_day_challenge/' directory)
6. **Daily Check-ins**: Each day, check in with the user to track progress and adapt the plan as needed (in the files you created (in '30_day_challenge') as well as the TODO list). Store daily notes in long-term memory.

## Important
- Always check user profile, preferences and previous daily notes first for context
- Respect any medical conditions or dietary restrictions
- Provide clear, actionable recommendations
- Save progress to files so users can reference later (always save to the '30_day_challenge/' directory, update the same file as needed to keep all information in one place)"""
)

In [46]:
# Step 4: Test with a user creating their 30-day challenge
TODO_STORE.clear()

result = wellness_coach_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. I'd like you to create a personalized 30-day wellness challenge for me.
I want to focus on improving my energy levels and sleep quality. I can dedicate 30 minutes each morning to this challenge. I prefer bodyweight exercises and I'm interested in mindfulness practices. Please create a comprehensive plan and save it as a file I can reference."""
    }]
})

print("Wellness Coach response:")
print(result["messages"][-1].content)

Wellness Coach response:
Hi Alex! I created your personalized 30-day Energy + Sleep Morning Challenge and saved it for you here:
- /30_day_challenge/30_day_energy_sleep_challenge_user_alex.md

What’s inside
- 30-minute morning block you can follow daily (bodyweight only), designed to boost daytime energy and support better sleep
- Weekly structure (4 weeks) with daily session types (Strength, Mobility + Conditioning, Recovery/Deload, Full Body, Balance, Cardio + Core)
- Detailed session templates with warm-up, main set, finisher, and cooldown (with options and progressions)
- Progressive overload plan (simple, trackable) with RPE guidance
- Mindfulness: brief morning breath + intention; optional evening wind-down menu (short or long) for sleep
- Modifications for stiff hips/hamstrings, sensitive wrists/shoulders, and limited space
- Tracking metrics and a 30-day checklist to log progress
- “Minimum Effective Dose” one-page version for busy days
- Vegetarian-friendly recovery tips and a

In [47]:
print(list_todos.invoke({}))

✅ [todo_1] Initial assessment: Retrieve profile, preferences, and prior daily notes for user_alex (completed)
✅ [todo_3] Planning: Draft 30-day wellness challenge outline (energy + sleep focus) (completed)
✅ [todo_5] Delegation: Physical specialist to create 30-min bodyweight morning program (completed)
✅ [todo_7] Delegation: Mental specialist to create mindfulness + sleep protocol (completed)
✅ [todo_9] Integration: Combine physical and mental plans into a cohesive program (completed)
✅ [todo_11] Documentation: Save integrated plan to /30_day_challenge/ as a reference file (completed)
✅ [todo_13] Set up daily check-in template and tracking metrics in the plan file (completed)
✅ [todo_15] Record a daily note summarizing plan creation and request start date/timezone (completed)
⬜ [todo_17] Ongoing: Daily check-ins for 30 days and adapt as needed (pending)


In [48]:
# Step 5: Simulate a daily check-in and adaptation
checkin_result = wellness_coach_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Day 1 check-in: I'm feeling good overall, but I'm struggling to stay consistent with the mindfulness practices. I find it hard to quiet my mind in the mornings. Can you help me adapt the plan to make it easier for me to engage with mindfulness? Maybe suggest some alternative techniques or resources."""
    }]
})

print("\nDaily Check-in Response:")
print(checkin_result["messages"][-1].content)


Daily Check-in Response:
Got it—thanks for the Day 1 check-in. It’s totally normal for “quiet mind” practices to feel tough in the morning. I’ve adapted your plan and saved it here: /30_day_challenge/30_day_energy_sleep_challenge_user_alex.md under “Mindfulness Adaptations (Day 1 Update — 2026-02-11).” I also logged your Day 1 notes.

Quick win for tomorrow morning (5 minutes, no silent sitting)
- 1 min: Breathe 4-in/6–8-out (nasal if comfy), soft gaze.
- 2 min: Gentle movement: reach + side bends, shoulder rolls, hip circles.
- 2 min: Sensory grounding: notice 1 thing you can see, hear, feel, smell, taste.

If you have 10 minutes
- 3 min: 4-in/6-out breathing
- 4 min: Dynamic stretch flow (neck rolls, shoulder shrugs, torso twists, calf raises)
- 3 min: Mindful listening (calm soundscape; notice 3 layers of sound)

On‑the‑Spot reset (3–5 minutes, anytime)
- 30–60s: 4-in/6–8-out breathing
- 2 min: Mini body scan (feet→head), soften jaw/shoulders
- 1–2 min: Calm-place visualization wit

In [50]:
# get user data to see the stored check-in notes
print("\nUser data (daily check-ins):")
print(get_user_data.invoke({"user_id": "user_alex", "data_type": "daily_checkins"}))


User data (daily check-ins):
Daily_checkins for user_alex:
  2026-02-11: {'value': 'Created and saved integrated 30-day Energy + Sleep Morning Challenge with 30-min bodyweight sessions and mindfulness practices. File: /30_day_challenge/30_day_energy_sleep_challenge_user_alex.md. Awaiting Alex’s start date and timezone. Asked to confirm any injuries or constraints beyond mild anxiety and vegetarian diet.'}


---
## Summary

In this session, we explored **Deep Agents** and their four key elements:

| Element | Purpose | Implementation |
|---------|---------|----------------|
| **Planning** | Track complex tasks | `write_todos`, `update_todo`, `list_todos` |
| **Context Management** | Handle large contexts | File system tools, automatic offloading |
| **Subagent Spawning** | Delegate to specialists | `task` tool with custom configs |
| **Long-term Memory** | Remember across sessions | LangGraph Store integration |

### Key Takeaways:

1. **Deep Agents handle complexity** - Unlike simple tool loops, they can manage long-horizon, multi-step tasks
2. **Planning is context engineering** - Todo lists and files aren't just organization—they're extended memory
3. **Subagents prevent context bloat** - Delegation keeps the main agent focused and efficient
4. **Skills enable progressive disclosure** - Load capabilities on-demand instead of upfront
5. **The CLI makes interaction natural** - Interactive sessions with conversation resume

### Deep Agents vs Traditional Agents

| Aspect | Traditional Agent | Deep Agent |
|--------|-------------------|------------|
| Task complexity | Simple, single-step | Complex, multi-step |
| Context management | All in conversation | Files + summaries |
| Delegation | None | Subagent spawning |
| Memory | Within thread | Across sessions |
| Planning | Implicit | Explicit (todos) |

### When to Use Deep Agents

**Use Deep Agents when:**
- Tasks require multiple steps or phases
- Context would overflow in a simple loop
- Specialization would improve quality
- Users need to resume sessions
- Long-term memory is valuable

**Use Simple Agents when:**
- Tasks are straightforward Q&A
- Single tool call suffices
- Context fits easily
- No need for persistence

### Further Reading

- [Deep Agents Documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents GitHub](https://github.com/langchain-ai/deepagents)
- [Context Management Blog Post](https://www.blog.langchain.com/context-management-for-deepagents/)
- [Building Multi-Agent Applications](https://www.blog.langchain.com/building-multi-agent-applications-with-deep-agents/)
- [LangGraph Memory Concepts](https://langchain-ai.github.io/langgraph/concepts/memory/)